# LOOKOUT VO — WaSR-SLR water masks for flir-VIS (Colab / Kaggle, GPU)

WaSR (`lojzezust/WaSR@fdb2b03`, Apache-2.0) with the SLR weights `wasr_slr_v2_rn101.pth` (sha256 `1e6cb14d…`), full resolution
(1920 px) on every 2nd ≈8 Hz VO frame of the flir-VIS clips; the class map is stored at 960 px (the tool's VO width). The VO tool
reads the maps with `LOOKOUT_WATER_MASK_DIR` and takes the water profile only from pixels of class water (`dk_slr2_fvt_raw`).

Default configuration = the input of the final entry: the 11 flir-VIS test clips.

**Runtime → GPU (T4)**, then *Run all*. `SMOKE = True`: one clip, 40 frames. Results on Google Drive
(`MyDrive/lookout/wasr_runs/<RUN_NAME>`); finished clips are skipped. Two sessions in parallel: same configuration,
`SHARD_COUNT = 2`, `SHARD_INDEX` 0 and 1. The last cells pack the results and write the environment lock file.

In [ ]:
# ============================ CONFIG ============================
WEIGHTS = 'slr'                  # 'slr' = wasr_slr_v2_rn101 (the final entry); 'mastr' = wasr_rn101
RUN_NAME = f'wasr_v4_{WEIGHTS}_flirvis'  # class map 960 px
PRIORITY = []                    # clips first, before GROUPS
GROUPS = ['test:flir-VIS']       # 'split:camera', '*' = every camera
CLIPS = []                       # explicit clip keys override PRIORITY and GROUPS
STEP = 2                         # every STEP-th ~8 Hz VO frame (2 = ~4 Hz); saved 'frames' are VO step indices
INFER_W = 1920                   # network input width (height keeps the aspect, multiple of 8); 1920 = full resolution
STORE_W = 960                    # 960 = the tool's VO width (lookout_vo.py resizes the map to it); width of the saved class map (nearest); INFER_W = full; both profiles are always at INFER_W
SAVE_SKYLINE = True              # False = the v2 file format (no full-res skyline; shore_course then takes it from the class map)
REVERSE = False                  # True: clips from the END of the order (a second session meeting the first in the middle)
BATCH = 2                        # frames per forward pass (T4, 1920 wide: 2)
SMOKE = False                    # True: first PRIORITY clip, 40 frames, nothing on Drive
SHARD_INDEX, SHARD_COUNT = 0, 1  # several notebooks on one RUN_NAME: same SHARD_COUNT, different SHARD_INDEX
SAVE_TO_DRIVE = True
DATA_URL = 'https://macvi.org/downloads/lookout/v0.1/lookout-vo-dev-v0.1.tar'
WASR_COMMIT = 'fdb2b036ab9e43bc92a44816852b667737ee79cd'
WASR_WEIGHTS = {'mastr': ('https://github.com/lojzezust/WaSR/releases/download/weights/wasr_rn101.pth', '5f54dec28a9eac45d3e277819ccf33290421395c7ecac52e1755e3e285c009b0'), 'slr': ('https://github.com/lojzezust/SLR/releases/download/weights_v2/wasr_slr_v2_rn101.pth', '1e6cb14df9129f80d426932a50068c0dc613ee91ad40fb89c78a8435f53ae74b')}[WEIGHTS]   # (url, sha256)

import os, sys, json, subprocess, hashlib, shutil, base64, gzip, time, zipfile, io, math
from pathlib import Path
if SMOKE:
    SAVE_TO_DRIVE = False; RUN_NAME += '_smoke'
SCRATCH = Path(os.environ.get('LOOKOUT_SCRATCH', '/content/lookout')); SCRATCH.mkdir(parents=True, exist_ok=True)
REL = Path(os.environ.get('LOOKOUT_RELEASE', SCRATCH / 'release')); MANIFEST = REL / 'release_manifest.json'
DRIVE = None
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/lookout'); DRIVE.mkdir(parents=True, exist_ok=True)
OUT = (DRIVE if DRIVE else SCRATCH) / 'wasr_runs' / RUN_NAME; OUT.mkdir(parents=True, exist_ok=True)

def sh_log(cmd, tail=60):
    """Run; on failure print the LAST lines of the output."""
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        print('\n'.join((r.stdout + r.stderr).splitlines()[-tail:]))
        raise RuntimeError(f'failed ({r.returncode}): {cmd}')
    return r.stdout

## 1 — Development data (6.8 GB; taken from Drive if already there)

In [ ]:
if not MANIFEST.exists():
    tar = (DRIVE / 'lookout-vo-dev-v0.1.tar') if DRIVE else (SCRATCH / 'lookout.tar')
    if not tar.exists():
        sh_log(f'curl -sSL -o {tar} {DATA_URL}')
    sh_log(f'tar -xf {tar} -C {SCRATCH}')
    if not DRIVE:
        tar.unlink()
sh_log(f'cd {REL} && sha256sum --quiet --check CHECKSUMS.sha256')
print({s: sum(c['split'] == s for c in json.load(open(MANIFEST))['clips']) for s in ('train', 'val')})

## 2 — Test data (1.4 GB, SHA-256 checked)

In [ ]:
# ---- held-out TEST package (video, calibration, frame times; no references), as in lookout_speed_v3_1_colab ----
TEST_URL = DATA_URL.replace('lookout-vo-dev-v0.1.tar', 'lookout-vo-test-v0.1.tar')
TREL = Path(os.environ.get('LOOKOUT_TEST_RELEASE', SCRATCH / 'lookout-vo-test-v0.1')); TMAN = TREL / 'release_manifest.json'
if not TMAN.exists():
    tar = (DRIVE / 'lookout-vo-test-v0.1.tar') if DRIVE else (SCRATCH / 'lookout-vo-test-v0.1.tar')
    if not tar.exists():
        sh_log(f'curl -sSL -o {tar} {TEST_URL}')
    want = sh_log(f'curl -sSL {TEST_URL}.sha256').split()[0]; got = sh_log(f'sha256sum {tar}').split()[0]
    assert got == want, f'test archive checksum {got} != published {want}'
    sh_log(f'tar -xf {tar} -C {SCRATCH}')
    if not DRIVE:
        tar.unlink()
    sh_log(f'cd {TREL} && sha256sum --quiet --check CHECKSUMS.sha256')
print(len(json.load(open(TMAN))['clips']), 'test clips')

## 3 — WaSR (code pinned, weights checked by sha256)

In [ ]:
# ---- WaSR: code pinned to a commit (Colab's own torch), weights checked by sha256 ----
import torch, torchvision
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEV == 'cpu':
    print('NO GPU -- Runtime -> Change runtime type -> GPU. (CPU only makes sense for SMOKE.)')
WASR_DIR = Path(os.environ.get('WASR_SRC', SCRATCH / 'WaSR'))
if not (WASR_DIR / 'wasr' / 'models.py').exists():
    sh_log(f'git clone -q https://github.com/lojzezust/WaSR.git {WASR_DIR}')
    sh_log(f'git -C {WASR_DIR} checkout -q {WASR_COMMIT}')
try:
    import pytorch_lightning                                       # wasr/utils.py imports it (callback class only)
except ImportError:
    sh_log(f'{sys.executable} -m pip install -q pytorch-lightning')
sys.path.insert(0, str(WASR_DIR))
import wasr.models as wasr_models
# wasr_deeplabv2_resnet101 builds its backbone with ImageNet weights (a download); all of them are overwritten below
wasr_models.resnet101 = lambda pretrained=False, **kw: torchvision.models.resnet.resnet101(weights=None, **kw)
WPTH = Path(os.environ.get('WASR_WEIGHTS_FILE', (DRIVE or SCRATCH) / WASR_WEIGHTS[0].rsplit('/', 1)[1]))
if not WPTH.exists():
    sh_log(f'curl -sSL -o {WPTH} {WASR_WEIGHTS[0]}')
got = hashlib.sha256(WPTH.read_bytes()).hexdigest()
assert got == WASR_WEIGHTS[1], f'weights sha256 {got} != {WASR_WEIGHTS[1]}'
model = wasr_models.get_model('wasr_resnet101', pretrained=False)
sd = torch.load(WPTH, map_location='cpu'); model.load_state_dict(sd.get('model', sd)); model = model.to(DEV).eval()
HALF = DEV == 'cuda'
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 3, 1, 1)   # WaSR datasets/transforms.py
STD = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 3, 1, 1)
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0) if DEV == 'cuda' else 'cpu')

def segment(batch):
    """batch B x H x W x 3 uint8 RGB -> (edge B x W float32: sub-pixel top row of the water (P(water) crosses 0.5
    going down), NaN where the column has no water or water starts at the top row; above B x W uint8: class 3 rows
    above the edge (0 obstacle/shore, 2 sky, 255 none); skyline B x W float32: sub-pixel top of the shore / buildings /
    quay (P(sky) drops below 0.5 going down), NaN where the column does not start in sky or the first non-sky pixel
    below is water (open horizon); cls B x H x W uint8 argmax: 0 obstacle, 1 water, 2 sky).
    Output upsampled to the input size (bilinear) as WaSR's Predictor."""
    x = torch.from_numpy(batch).to(DEV).permute(0, 3, 1, 2).float().div_(255).sub_(MEAN).div_(STD)
    with torch.no_grad():
        with torch.autocast('cuda', dtype=torch.float16, enabled=HALF):
            out = model({'image': x})['out']
        p = torch.nn.functional.interpolate(out.float(), x.shape[-2:], mode='bilinear', align_corners=False).softmax(1)
        cls = p.argmax(1).to(torch.uint8); pw = p[:, 1]                  # B x H x W
        w = pw > 0.5; has = w.any(1); r = w.to(torch.uint8).argmax(1)   # first water row per column
        rm = (r - 1).clamp(min=0); g = lambda rr: pw.gather(1, rr[:, None]).squeeze(1)
        a, b = g(rm), g(r); frac = ((0.5 - a) / (b - a).clamp(min=1e-6)).clamp(0, 1)
        edge = torch.where(has & (r >= 3), rm.float() + frac, torch.full_like(frac, float('nan')))
        ra = (r - 3).clamp(min=0); above = cls.gather(1, ra[:, None]).squeeze(1)
        above = torch.where(has & (r >= 3), above, torch.full_like(above, 255))
        ps = p[:, 2]; ns = ps < 0.5; hs = ns.any(1) & ~ns[:, 0]; q = ns.to(torch.uint8).argmax(1)   # first non-sky row
        qm = (q - 1).clamp(min=0); gs = lambda rr: ps.gather(1, rr[:, None]).squeeze(1)
        a, b = gs(qm), gs(q); frac = ((a - 0.5) / (a - b).clamp(min=1e-6)).clamp(0, 1)
        qb = (q + 3).clamp(max=ps.shape[1] - 1); below = cls.gather(1, qb[:, None]).squeeze(1)
        skyline = torch.where(hs & (below == 0), qm.float() + frac, torch.full_like(frac, float('nan')))
    return edge.cpu().numpy(), above.cpu().numpy(), skyline.cpu().numpy(), cls.cpu().numpy()

## 4 — Run (resumes after a disconnect)

In [ ]:
# ---- decode each clip once (starter decoder, ~8 Hz, FULL resolution), WaSR on every STEP-th frame ----
import numpy as np, cv2
sys.path.insert(0, str(REL / 'starter'))
from preprocessing import ClipPreprocessor
from public_dataset import load_timestamps
MAN = {c['key']: (REL, c) for c in json.load(open(MANIFEST))['clips']}
MAN.update({c['key']: (TREL, c) for c in json.load(open(TMAN))['clips']})
def selection():
    if CLIPS:
        return list(CLIPS)
    out = [k for k in PRIORITY if k in MAN]
    for g in GROUPS:
        sp, cam = g.split(':')
        out += [k for k in sorted(MAN) if MAN[k][1]['split'] == sp and cam in ('*', MAN[k][1]['camera_mode']) and k not in out]
    return out[::-1] if REVERSE else out
def run_clip(k):
    root, c = MAN[k]; t0 = time.time(); tn = 0.0
    pre = ClipPreprocessor(str(root / c['video']), c['camera_mode'], 8.0, half_res=False, calibration=c['calibration'],
                           frame_timestamps=load_timestamps(str(root), c))
    fr, ts, E, A, SL, CL, buf, times = [], [], [], [], [], [], [], []
    def flush():
        nonlocal tn
        t1 = time.time(); e, a, sk, cl = segment(np.stack([b for _, _, b in buf])); tn += time.time() - t1
        for (i, t, _), ee, aa, ss, cc in zip(buf, e, a, sk, cl):
            fr.append(i); ts.append(t); E.append(ee.astype(np.float16)); A.append(aa); SL.append(ss.astype(np.float16))
            CL.append(cc if STORE_W == cc.shape[1] else
                      cv2.resize(cc, (STORE_W, round(cc.shape[0] * STORE_W / cc.shape[1])), interpolation=cv2.INTER_NEAREST))
        buf.clear()
    # decoding (CPU) runs in a thread while WaSR (GPU) works on the previous batch -- same frames, same order
    import threading, queue
    Q = queue.Queue(maxsize=4 * BATCH); stop = threading.Event(); meta = {}
    def produce():
        try:
            for i, (_idx, image, intr, pts, decoded) in enumerate(pre.frames_with_time()):
                if stop.is_set():
                    break
                times.append(float(pts))
                if i % STEP:
                    continue
                if 'size' not in meta:
                    h, w = image.shape[:2]
                    meta.update(size=(INFER_W, int(round(h * INFER_W / w / 8)) * 8), hw=(h, w), intr0=np.asarray(intr[:4], float))
                img = image if image.shape[1::-1] == meta['size'] else cv2.resize(image, meta['size'], interpolation=cv2.INTER_AREA)
                Q.put((i, float(pts), cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))
        except BaseException as exc:                                # surfaced in the main thread
            meta['error'] = exc
        finally:
            Q.put(None)
    th = threading.Thread(target=produce, daemon=True); th.start()
    while True:
        item = Q.get()
        if item is None:
            break
        buf.append(item)
        if len(buf) == BATCH:
            flush()
        if SMOKE and len(fr) + len(buf) >= 40:
            stop.set()
            while Q.get() is not None:                              # drain so the producer can finish
                pass
            break
    th.join()
    if 'error' in meta:
        raise meta['error']
    size, (h0, w0), intr0 = meta['size'], meta['hw'], meta['intr0']
    if buf:
        flush()
    sx, sy = size[0] / w0, size[1] / h0                               # K of the saved edges = the frame's K scaled
    K = np.array([intr0[0] * sx, intr0[1] * sy, intr0[2] * sx, intr0[3] * sy])
    dst = OUT / f'{k}.npz'; tmp = OUT / f'{k}.tmp.npz'
    np.savez_compressed(tmp, frames=np.asarray(fr), t=np.asarray(ts), times=np.asarray(times), edge=np.asarray(E),
                        above=np.asarray(A), **({'skyline': np.asarray(SL)} if SAVE_SKYLINE else {}), cls=np.asarray(CL), K=K, frame_wh=np.asarray([w0, h0]),
                        infer_wh=np.asarray(size), step=STEP, smoke=SMOKE, wasr=WASR_COMMIT, weights=WASR_WEIGHTS[1],
                        camera_mode=c['camera_mode'], split=c['split'])
    os.replace(tmp, dst)
    print(f'{k} {c["camera_mode"]:8s} {c["split"]:5s} {len(fr)} frames {size[0]}x{size[1]}, total {time.time() - t0:5.0f}s, '
          f'WaSR {tn:5.0f}s ({len(fr) / max(tn, 1e-9):.2f} fr/s), {dst.stat().st_size / 1e6:.2f} MB', flush=True)
sel = selection()
if SMOKE:
    sel = sel[:1]
sel = sel[SHARD_INDEX::SHARD_COUNT]
done = lambda k: (OUT / f'{k}.npz').exists()
todo = [k for k in sel if not done(k)]
print(f'{RUN_NAME}: {len(sel)} clips in this shard, {len(todo)} to run')
log = open(OUT / f'log_shard{SHARD_INDEX}.txt', 'a')
for k in todo:
    if done(k):                                                    # another session (other CLIPS order) finished it meanwhile
        continue
    try:
        run_clip(k); log.write(f'{k} ok\n')
    except Exception as exc:                                       # one bad clip must not stop the rest
        print(k, 'FAILED', repr(exc)); log.write(f'{k} FAILED {exc!r}\n')
    log.flush()
print(sum(done(k) for k in sel), '/', len(sel), 'clips done')

In [ ]:
# ---- quick look: share of frames whose water edge / skyline covers >= 40% of the width ----
for k in sel:
    f = OUT / f'{k}.npz'
    if not f.exists():
        continue
    z = np.load(f); ok = np.isfinite(z['edge'].astype(float)); shore = (z['above'] == 0) & ok
    print(f'{k} {str(z["camera_mode"]):8s} frames {len(z["frames"]):5d}  edge found {ok.mean():.2f}  '
          f'frames with shore >= 40% of width {(shore.mean(1) >= 0.4).mean():.2f}  '
          + (f'with skyline >= 40% {(np.isfinite(z["skyline"].astype(float)).mean(1) >= 0.4).mean():.2f}' if 'skyline' in z else ''))

## 5 — Package of the results

In [ ]:
# ---- the package to send back: every finished npz + logs ----
zp = OUT / f'{RUN_NAME}_shard{SHARD_INDEX}.zip'; n = 0
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_STORED) as z:            # npz already compressed
    for f in sorted(OUT.glob('clip_*.npz')):
        if not f.name.endswith('.tmp.npz'):
            z.write(f, f.name); n += 1
    for f in OUT.glob('log_shard*.txt'):
        z.write(f, f'logs/{f.name}')
    z.writestr('README.txt', f'{RUN_NAME} shard {SHARD_INDEX}/{SHARD_COUNT}; STEP {STEP}; INFER_W {INFER_W}; STORE_W {STORE_W}; '
               f'WaSR {WASR_COMMIT}; weights {WEIGHTS} {WASR_WEIGHTS[0]} sha256 {WASR_WEIGHTS[1]}\n')
print(f'{zp}: {n} clips, {zp.stat().st_size / 1e6:.1f} MB  <- this file goes back')

In [ ]:
# ---- environment lock of this run (next to the results on Drive) ----
import sys, platform, subprocess, pathlib
L = OUT / f'env_lock_{RUN_NAME}.txt'
L.write_text(f"python {sys.version}\nplatform {platform.platform()}\n"
             + subprocess.run(f"{sys.executable} -m pip freeze", shell=True, capture_output=True, text=True).stdout
             + subprocess.run("nvidia-smi --query-gpu=name,driver_version --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)
import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'cudnn', torch.backends.cudnn.version(), file=open(L, 'a'))
print(L, '\n', L.read_text()[:1500])
